# Tiền xử lý dữ liệu cho mô hình RNN

Notebook này thực hiện toàn bộ quá trình tiền xử lý dữ liệu nhằm chuyển các file văn bản đã làm sạch thành dữ liệu số mà mô hình RNN có thể sử dụng để huấn luyện. Kết quả cuối cùng được lưu vào một file duy nhất để phục vụ các bước xây dựng mô hình.

---

## Tại sao chọn phụ đề phim *Young Sheldon* làm tập dữ liệu?

### 1. Ngôn ngữ giao tiếp tự nhiên
Phim chứa nhiều đoạn hội thoại đời thường, giúp mô hình học được cách sử dụng ngôn ngữ giống như con người giao tiếp trong thực tế.

### 2. Từ vựng đa dạng
Bộ phim kết hợp giữa các cuộc trò chuyện gia đình đơn giản và những thuật ngữ khoa học của Sheldon, giúp dữ liệu phong phú và tăng khả năng học ngôn ngữ của mô hình.

### 3. Khối lượng dữ liệu phù hợp
22 tập phim cung cấp hơn 11.000 câu thoại, đủ lớn để huấn luyện mô hình hiệu quả nhưng không quá lớn gây quá tải cho máy tính cá nhân.

### Kết luận
*Young Sheldon* là nguồn dữ liệu phù hợp vì có ngôn ngữ tự nhiên, từ vựng đa dạng và kích thước dữ liệu vừa phải cho việc xây dựng mô hình dự đoán từ tiếp theo bằng RNN.

## 1. Thiết lập các tham số

Đầu tiên, notebook khai báo một số **hyper-parameters** dùng để kiểm soát quá trình tiền xử lý dữ liệu.

- **`MIN_WORD_LEN = 5`**
  - Chỉ giữ lại các câu có tối thiểu **5 từ**.

- **`MAX_WORD_LEN = 30`**
  - Chỉ giữ lại các câu có tối đa **30 từ**.

- **`SEQ_LENGTH = 15`**
  - Độ dài của chuỗi đầu vào khi tạo dữ liệu theo phương pháp **Sliding Window**.

- **`MIN_WORD_FREQ = 3`**
  - Những từ xuất hiện ít hơn **3 lần** trong toàn bộ tập huấn luyện sẽ được xem là từ hiếm và thay thế bằng token `<unk>`.

Các tham số này giúp loại bỏ dữ liệu không cần thiết, giảm kích thước từ điển và cải thiện khả năng học của mô hình.



In [1]:
import os
import re
import json
import glob
from collections import Counter

# Thư mục chứa dữ liệu đã làm sạch
CLEANED_DIR = "Cleaned_Text"

# Thư mục lưu dữ liệu sau tiền xử lý NLP
OUTPUT_DIR = "Preprocessed_Data"

# Tạo thư mục đầu ra nếu chưa tồn tại
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Các tham số cho quá trình chuẩn bị dữ liệu RNN

MIN_WORD_LEN = 5      # Độ dài từ tối thiểu
MAX_WORD_LEN = 30     # Độ dài từ tối đa

SEQ_LENGTH = 15       # Số từ trong một chuỗi đầu vào

MIN_WORD_FREQ = 3     # Tần suất xuất hiện tối thiểu của từ

# Thông báo cấu hình hoàn tất
print("NLP Preprocessing Configured.")

NLP Preprocessing Configured.


## 2. Tokenization – Tách câu thành các token

Máy tính không làm việc trực tiếp với một câu hoàn chỉnh mà cần chia câu thành các đơn vị nhỏ hơn gọi là **token**.

Hàm `tokenize()` thực hiện hai bước:

- Chuyển toàn bộ ký tự sang **chữ thường**.
- Tách câu thành từng token, bao gồm cả từ và dấu câu.

Ví dụ:

```text
"I don't know, Shelly!"
```

Sau khi tokenization:

```text
['i', "don't", 'know', ',', 'shelly', '!']
```

---


In [2]:
def tokenize(text):
    # Chuyển toàn bộ văn bản thành chữ thường
    text = text.lower()

    # Tách văn bản thành các token (từ và dấu câu)
    tokens = re.findall(r"[a-z0-9]+(?:'[a-z]+)?|[^\n\w\s]", text)

    # Loại bỏ token rỗng
    return [t for t in tokens if t.strip()]

# Ví dụ kiểm tra hàm
sample = "I don't know, Shelly! (train whistle blows) Are we going?"

# In văn bản gốc
print("Sample raw text:", sample)

# In danh sách token sau khi tách
print("Tokenized:", tokenize(sample))


Sample raw text: I don't know, Shelly! (train whistle blows) Are we going?
Tokenized: ['i', "don't", 'know', ',', 'shelly', '!', '(', 'train', 'whistle', 'blows', ')', 'are', 'we', 'going', '?']


## 3. Chia tập dữ liệu

Sau khi đọc các file `.txt` đã được làm sạch, dữ liệu được chia thành ba tập:

| Tập dữ liệu | Các tập phim | Mục đích |
|-------------|-------------|----------|
| **Train** | Episode 1 – 14 | Huấn luyện mô hình |
| **Validation (Val)** | Episode 15 – 18 | Đánh giá trong quá trình huấn luyện |
| **Test** | Episode 19 – 22 | Đánh giá cuối cùng |

Việc chia dữ liệu giúp đảm bảo mô hình được đánh giá khách quan và tránh hiện tượng học thuộc dữ liệu.



# Chia tập dữ liệu

Để đánh giá mô hình một cách khách quan, bộ dữ liệu được chia thành 3 phần:

* **Train Set (Tập 1–14):** Dùng để huấn luyện mô hình, giúp AI học các quy luật ngôn ngữ từ dữ liệu.
* **Validation Set (Tập 15–18):** Dùng để kiểm tra và điều chỉnh mô hình trong quá trình huấn luyện.
* **Test Set (Tập 19–22):** Dùng để đánh giá kết quả cuối cùng trên dữ liệu mà mô hình chưa từng thấy.

## Mục đích

Việc chia dữ liệu thành Train, Validation và Test giúp đánh giá mô hình một cách công bằng, khách quan và hạn chế hiện tượng **học thuộc dữ liệu (Overfitting)**, từ đó phản ánh chính xác khả năng dự đoán của mô hình trên dữ liệu mới.


In [3]:
# Chia dữ liệu thành Train, Validation và Test
train_episodes = [f"S01E{i:02d}" for i in range(1, 15)]
val_episodes = [f"S01E{i:02d}" for i in range(15, 19)]
test_episodes = [f"S01E{i:02d}" for i in range(19, 23)]

def load_split_data(episodes):
    # Lưu các câu thoại của tập dữ liệu
    sentences = []

    # Đọc từng tập phim
    for ep in episodes:
        filepath = os.path.join(CLEANED_DIR, f"{ep}.txt")

        # Bỏ qua nếu không tìm thấy file
        if not os.path.exists(filepath):
            print(f"Warning: File {filepath} not found. Skipping.")
            continue

        # Đọc nội dung file
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()

                # Chỉ lấy các dòng không rỗng
                if line:
                    sentences.append(line)

    return sentences

# Nạp dữ liệu Train
train_raw = load_split_data(train_episodes)

# Nạp dữ liệu Validation
val_raw = load_split_data(val_episodes)

# Nạp dữ liệu Test
test_raw = load_split_data(test_episodes)

# Hiển thị số lượng câu thoại của từng tập dữ liệu
print(f"Loaded Raw Sentences: Train={len(train_raw)}, Val={len(val_raw)}, Test={len(test_raw)}")


Loaded Raw Sentences: Train=6072, Val=1731, Test=1839


## 4. Lọc các câu không phù hợp

Notebook tiếp tục loại bỏ những câu có độ dài không đáp ứng yêu cầu.

- Những câu **ngắn hơn 5 từ** sẽ bị loại bỏ vì không cung cấp đủ ngữ cảnh để dự đoán từ tiếp theo.

- Những câu **dài hơn 30 từ** cũng bị loại bỏ nhằm giảm độ phức tạp của chuỗi đầu vào và hạn chế ảnh hưởng của hiện tượng mất thông tin trong RNN.

Sau bước này, chỉ các câu có độ dài phù hợp mới được giữ lại.


In [4]:
# Tokenize và lọc câu theo độ dài quy định
def process_sentences(raw_sentences):
    processed = []

    # Duyệt từng câu thoại
    for s in raw_sentences:
        tokens = tokenize(s)

        # Chỉ giữ các câu có từ 5 đến 30 token
        if MIN_WORD_LEN <= len(tokens) <= MAX_WORD_LEN:
            processed.append(tokens)

    return processed

# Tiền xử lý dữ liệu Train
train_tokenized = process_sentences(train_raw)

# Tiền xử lý dữ liệu Validation
val_tokenized = process_sentences(val_raw)

# Tiền xử lý dữ liệu Test
test_tokenized = process_sentences(test_raw)

# Thống kê sau khi lọc độ dài câu
print(f"After length filtering ({MIN_WORD_LEN}-{MAX_WORD_LEN} tokens):")

print(f"Train sentences: {len(train_tokenized)} (filtered out {len(train_raw) - len(train_tokenized)})")
print(f"Val sentences: {len(val_tokenized)} (filtered out {len(val_raw) - len(val_tokenized)})")
print(f"Test sentences: {len(test_tokenized)} (filtered out {len(test_raw) - len(test_tokenized)})")


After length filtering (5-30 tokens):
Train sentences: 4700 (filtered out 1372)
Val sentences: 1319 (filtered out 412)
Test sentences: 1428 (filtered out 411)


## 5. Xây dựng từ điển (Vocabulary)
Đây là bước chuyển đổi dữ liệu văn bản sang dạng mà máy tính có thể xử lý.

Notebook đếm tần suất xuất hiện của tất cả các từ trong **tập Train**, sau đó gán cho mỗi từ một **ID** duy nhất.

Ví dụ:

| Từ | ID |
|-----|----|
| ! | 4 |
| always | 54 |
| train | 1084 |

Ngoài ra, hệ thống tạo thêm bốn **Special Tokens**:

| Token | Ý nghĩa |
|--------|----------|
| `<pad>` | Đệm các câu về cùng độ dài |
| `<unk>` | Đại diện cho từ không có trong từ điển |
| `<bos>` | Đánh dấu bắt đầu câu |
| `<eos>` | Đánh dấu kết thúc câu |

Việc xây dựng từ điển **chỉ sử dụng tập Train**, giúp tránh hiện tượng mô hình biết trước từ vựng xuất hiện trong tập Validation hoặc Test.



In [5]:
# Xây dựng từ điển từ dữ liệu Train
word_counter = Counter()

# Đếm tần suất xuất hiện của từng token
for tokens in train_tokenized:
    word_counter.update(tokens)

# Chỉ giữ các từ xuất hiện đủ số lần quy định
vocab_list = [word for word, count in word_counter.items()
              if count >= MIN_WORD_FREQ]

# Thêm các token đặc biệt
special_tokens = ["<pad>", "<unk>", "<bos>", "<eos>"]

# Ánh xạ từ -> chỉ số
word2idx = {token: idx for idx, token in enumerate(special_tokens)}

# Thêm các từ trong từ điển
for idx, word in enumerate(sorted(vocab_list)):
    word2idx[word] = len(word2idx)

# Ánh xạ chỉ số -> từ
idx2word = {idx: token for token, idx in word2idx.items()}

# Kích thước từ điển
vocab_size = len(word2idx)

# Hiển thị thông tin từ điển
print(f"Vocabulary built: {vocab_size} unique tokens (including special tokens)")
print("Special token indices:", {tok: word2idx[tok] for tok in special_tokens})
print("Sample vocab mapping:", list(word2idx.items())[4:15])

Vocabulary built: 1221 unique tokens (including special tokens)
Special token indices: {'<pad>': 0, '<unk>': 1, '<bos>': 2, '<eos>': 3}
Sample vocab mapping: [('!', 4), ('"', 5), ('$', 6), ('%', 7), ('&', 8), ("'", 9), (',', 10), ('-', 11), ('.', 12), ('00', 13), ('000', 14)]


In [6]:
# Lưu từ điển vào file JSON
vocab_data = {
    "word2idx": word2idx,  # Từ -> chỉ số
    "idx2word": idx2word   # Chỉ số -> từ
}

# Ghi dữ liệu từ điển ra file
with open(os.path.join(OUTPUT_DIR, "vocab.json"), 'w', encoding='utf-8') as f:
    json.dump(vocab_data, f, indent=4, ensure_ascii=False)

# Thông báo lưu thành công
print(f"Saved vocabulary to {os.path.join(OUTPUT_DIR, 'vocab.json')}")

Saved vocabulary to Preprocessed_Data\vocab.json


## 6. Chuyển từ thành chỉ số (Indexing)

Sau khi có từ điển, notebook chuyển toàn bộ token sang các chỉ số tương ứng.

Ví dụ:

Trước khi chuyển đổi:

```text
['i', 'always', 'loved', 'trains', '.']
```

Sau khi chuyển đổi:

```text
[525, 54, 624, 1084, 12]
```

Nếu gặp một từ không tồn tại trong từ điển, hệ thống sẽ thay thế bằng token `<unk>`.



In [7]:
# Chuyển token thành chỉ số theo từ điển
def map_to_indices(token_seqs):
    # ID của token không có trong từ điển
    unk_id = word2idx["<unk>"]

    # Thay mỗi token bằng chỉ số tương ứng
    return [[word2idx.get(tok, unk_id) for tok in seq] for seq in token_seqs]

# Chuyển dữ liệu Train sang dạng số
train_indexed = map_to_indices(train_tokenized)

# Chuyển dữ liệu Validation sang dạng số
val_indexed = map_to_indices(val_tokenized)

# Chuyển dữ liệu Test sang dạng số
test_indexed = map_to_indices(test_tokenized)

# Hiển thị ví dụ chuyển đổi
print("Example indexed sentence:")
print("Tokens:", train_tokenized[0])
print("Indices:", train_indexed[0])

Example indexed sentence:
Tokens: ["i've", 'always', 'loved', 'trains', '.']
Indices: [525, 54, 624, 1084, 12]


## 7. Tạo dữ liệu huấn luyện
Để mô hình RNN học nhiệm vụ **dự đoán từ tiếp theo**, notebook tạo hai định dạng dữ liệu khác nhau.


### 7.1. Định dạng theo từng câu

Mỗi câu được xử lý độc lập.

Quy trình gồm:

- Thêm token `<bos>` vào đầu câu.
- Thêm token `<eos>` vào cuối câu.
- Đệm thêm `<pad>` để tất cả các câu có cùng độ dài.

Ví dụ:

**Input**

```text
<bos> Sheldon loves trains <pad> <pad> ...
```

**Target**

```text
Sheldon loves trains <eos> <pad> <pad> ...
```

Trong đó, Target luôn dịch sang phải một vị trí so với Input để mô hình học dự đoán từ kế tiếp.

---

In [8]:
# FORMAT 1: Dataset theo từng câu thoại

# Lấy ID của các token đặc biệt
bos_id = word2idx["<bos>"]   # Bắt đầu câu
eos_id = word2idx["<eos>"]   # Kết thúc câu
pad_id = word2idx["<pad>"]   # Đệm độ dài
unk_id = word2idx["<unk>"]   # Từ không xác định

# Tìm độ dài câu lớn nhất trong tập Train
max_seq_len = max(len(seq) for seq in train_indexed) + 1

def prepare_sentence_dataset(indexed_seqs, max_len):
    inputs = []
    targets = []

    # Duyệt từng câu
    for seq in indexed_seqs:

        # Input bắt đầu bằng <bos>
        inp_seq = [bos_id] + seq

        # Target kết thúc bằng <eos>
        targ_seq = seq + [eos_id]

        # Tính số lượng token cần đệm
        pad_len = max_len - len(inp_seq)

        if pad_len > 0:
            # Đệm <pad> cho đủ độ dài
            inp_seq += [pad_id] * pad_len
            targ_seq += [pad_id] * pad_len
        else:
            # Cắt bớt nếu vượt quá độ dài tối đa
            inp_seq = inp_seq[:max_len]
            targ_seq = targ_seq[:max_len]

        inputs.append(inp_seq)
        targets.append(targ_seq)

    return inputs, targets

# Tạo dataset cho Train
train_sent_in, train_sent_targ = prepare_sentence_dataset(
    train_indexed, max_seq_len
)

# Tạo dataset cho Validation
val_sent_in, val_sent_targ = prepare_sentence_dataset(
    val_indexed, max_seq_len
)

# Tạo dataset cho Test
test_sent_in, test_sent_targ = prepare_sentence_dataset(
    test_indexed, max_seq_len
)

# Hiển thị thông tin dataset
print(f"Sentence format dataset built with max_len={max_seq_len}:")
print(f"Train size: {len(train_sent_in)}")
print("Sample Input: ", train_sent_in[0])
print("Sample Target:", train_sent_targ[0])


Sentence format dataset built with max_len=20:
Train size: 4700
Sample Input:  [2, 525, 54, 624, 1084, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Sample Target: [525, 54, 624, 1084, 12, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


### 7.2. Định dạng Sliding Window

Khác với phương pháp trên, toàn bộ lời thoại trong một tập phim được nối thành một chuỗi liên tục.

Sau đó, một cửa sổ có độ dài:

```text
SEQ_LENGTH = 15
```

được trượt dần trên chuỗi để tạo dữ liệu huấn luyện.

Ví dụ với chuỗi:

```text
A B C D E F
```

và cửa sổ có độ dài **3**:

Lần thứ nhất:

```text
Input  = [A, B, C]
Target = [B, C, D]
```

Lần thứ hai:

```text
Input  = [B, C, D]
Target = [C, D, E]
```

Phương pháp này giúp mô hình học được ngữ cảnh liên tục thay vì chỉ giới hạn trong từng câu riêng lẻ.

In [9]:
# FORMAT 2: Dataset theo Sliding Window

def prepare_sliding_window(indexed_seqs, seq_len):

    # Ghép tất cả câu thành một chuỗi token liên tục
    flat_stream = []

    for seq in indexed_seqs:
        flat_stream += [bos_id] + seq + [eos_id]

    inputs = []
    targets = []

    # Tạo các cửa sổ trượt độ dài seq_len
    for i in range(0, len(flat_stream) - seq_len, 1):

        chunk = flat_stream[i : i + seq_len + 1]

        # Input gồm seq_len token đầu
        inputs.append(chunk[:-1])

        # Target là chuỗi dịch sang phải 1 token
        targets.append(chunk[1:])

    return inputs, targets, len(flat_stream)

# Tạo dataset Sliding Window cho Train
train_win_in, train_win_targ, train_total_tokens = prepare_sliding_window(
    train_indexed, SEQ_LENGTH
)

# Tạo dataset Sliding Window cho Validation
val_win_in, val_win_targ, val_total_tokens = prepare_sliding_window(
    val_indexed, SEQ_LENGTH
)

# Tạo dataset Sliding Window cho Test
test_win_in, test_win_targ, test_total_tokens = prepare_sliding_window(
    test_indexed, SEQ_LENGTH
)

# Hiển thị thông tin dataset
print(f"Sliding window format dataset built with seq_length={SEQ_LENGTH}:")
print(f"Train: {len(train_win_in)} sequences (flat stream size: {train_total_tokens} tokens)")
print(f"Val: {len(val_win_in)} sequences (flat stream size: {val_total_tokens} tokens)")
print(f"Test: {len(test_win_in)} sequences (flat stream size: {test_total_tokens} tokens)")

print("Sample Input: ", train_win_in[0])
print("Sample Target:", train_win_targ[0])


Sliding window format dataset built with seq_length=15:
Train: 47401 sequences (flat stream size: 47416 tokens)
Val: 13383 sequences (flat stream size: 13398 tokens)
Test: 14442 sequences (flat stream size: 14457 tokens)
Sample Input:  [2, 525, 54, 624, 1084, 12, 3, 2, 531, 351, 10, 528, 700, 1, 3]
Sample Target: [525, 54, 624, 1084, 12, 3, 2, 531, 351, 10, 528, 700, 1, 3, 2]


## 8. Lưu dữ liệu sau tiền xử lý

Sau khi hoàn thành toàn bộ các bước trên, notebook lưu:

- Vocabulary
- Special Tokens
- Hyper-parameters
- Dữ liệu Train
- Dữ liệu Validation
- Dữ liệu Test
- Hai định dạng Input–Target

vào một file duy nhất:

```text
preprocessed_data.json
```

File này là đầu vào trực tiếp cho mô hình RNN ở các bước huấn luyện sau, giúp quá trình xây dựng mô hình trở nên thuận tiện và không cần thực hiện lại các bước tiền xử lý.

In [10]:
# Đóng gói toàn bộ dữ liệu đã tiền xử lý
final_dataset = {
    "metadata": {
        # Thông tin cấu hình dữ liệu
        "vocab_size": vocab_size,
        "max_sentence_seq_len": max_seq_len,
        "sliding_window_seq_len": SEQ_LENGTH,
        "min_word_freq": MIN_WORD_FREQ,

        # ID của các token đặc biệt
        "special_tokens": {
            "pad": pad_id,
            "unk": unk_id,
            "bos": bos_id,
            "eos": eos_id
        }
    },

    # Dataset theo định dạng câu hoàn chỉnh
    "sentence_format": {
        "train": {"inputs": train_sent_in, "targets": train_sent_targ},
        "val": {"inputs": val_sent_in, "targets": val_sent_targ},
        "test": {"inputs": test_sent_in, "targets": test_sent_targ}
    },

    # Dataset theo định dạng Sliding Window
    "sliding_window_format": {
        "train": {"inputs": train_win_in, "targets": train_win_targ},
        "val": {"inputs": val_win_in, "targets": val_win_targ},
        "test": {"inputs": test_win_in, "targets": test_win_targ}
    }
}

# Đường dẫn file đầu ra
output_json_path = os.path.join(OUTPUT_DIR, "preprocessed_data.json")

# Lưu dữ liệu vào file JSON
with open(output_json_path, 'w', encoding='utf-8') as f:
    json.dump(final_dataset, f, indent=2, ensure_ascii=False)

# Thông báo hoàn thành
print(f"All preprocessed datasets saved successfully to: {output_json_path}")


All preprocessed datasets saved successfully to: Preprocessed_Data\preprocessed_data.json
